In [1]:
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

# Rutas
DATA_DIR     = Path(".")
YA_FILE      = DATA_DIR / "goodreads_reviews_young_adult.json"
SPOILER_FILE = DATA_DIR / "goodreads_reviews_spoiler_raw.json"

# Parámetros de muestreo
N_SAMPLE    = 100_000   # para MLP y BiLSTM
RANDOM_SEED = 42

print("Librerías cargadas correctamente ✓")
print(f"Muestra objetivo: {N_SAMPLE:,} reseñas")

Librerías cargadas correctamente ✓
Muestra objetivo: 100,000 reseñas


In [2]:
# Cargar dataset completo de Young Adult
print("Cargando dataset Young Adult completo (esto tarda unos minutos)...")

records = []
with open(YA_FILE, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Leyendo reseñas"):
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"\nTotal reseñas cargadas: {len(df):,}")
print(f"Columnas: {list(df.columns)}")

Cargando dataset Young Adult completo (esto tarda unos minutos)...


Leyendo reseñas: 0it [00:00, ?it/s]


Total reseñas cargadas: 2,389,900
Columnas: ['user_id', 'book_id', 'review_id', 'rating', 'review_text', 'date_added', 'date_updated', 'read_at', 'started_at', 'n_votes', 'n_comments']


In [3]:
# Función de limpieza de texto
def limpiar_texto(texto):
    if not isinstance(texto, str):
        return ""
    # Eliminar HTML
    texto = re.sub(r"<[^>]+>", " ", texto)
    # Eliminar saltos de línea y tabs
    texto = re.sub(r"[\n\r\t]", " ", texto)
    # Eliminar caracteres especiales pero mantener puntuación básica
    texto = re.sub(r"[^\w\s\.\,\!\?\'\"\-]", " ", texto)
    # Eliminar espacios múltiples
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

# Aplicar limpieza
print("Aplicando limpieza de texto...")
df["review_text"] = df["review_text"].apply(limpiar_texto)

# Filtrar registros inválidos
print("Filtrando registros inválidos...")
df = df[df["rating"] > 0]           # eliminar rating=0
df = df[df["review_text"].str.len() >= 50]  # mínimo 50 caracteres

print(f"\nReseñas después de limpieza: {len(df):,}")
print(f"Distribución de ratings:")
print(df["rating"].value_counts().sort_index())

Aplicando limpieza de texto...
Filtrando registros inválidos...

Reseñas después de limpieza: 2,053,504
Distribución de ratings:
rating
1     62464
2    160595
3    430459
4    703383
5    696603
Name: count, dtype: int64


In [5]:
# Muestreo estratificado por rating (compatible con pandas 3.0)
print("Aplicando muestreo estratificado...")

muestras_por_clase = N_SAMPLE // 5

piezas = []
for rating in [1, 2, 3, 4, 5]:
    subset = df[df["rating"] == rating]
    n = min(muestras_por_clase, len(subset))
    piezas.append(subset.sample(n=n, random_state=RANDOM_SEED))

df_sample = pd.concat(piezas).reset_index(drop=True)

print(f"Total muestra: {len(df_sample):,}")
print(f"\nDistribución por rating:")
print(df_sample["rating"].value_counts().sort_index())

Aplicando muestreo estratificado...
Total muestra: 100,000

Distribución por rating:
rating
1    20000
2    20000
3    20000
4    20000
5    20000
Name: count, dtype: int64


In [6]:
# Cargar dataset de spoilers y extraer has_spoiler
print("Cargando dataset de spoilers...")

records_sp = []
with open(SPOILER_FILE, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Leyendo spoilers"):
        records_sp.append(json.loads(line))

df_sp_full = pd.DataFrame(records_sp)

# Extraer has_spoiler del texto
df_sp_full["has_spoiler"] = df_sp_full["review_text"].str.contains(
    r"\(hide spoiler\)", case=False, regex=True, na=False
)

# Quedarnos solo con review_id y has_spoiler para el cruce
df_sp_labels = df_sp_full[["review_id", "has_spoiler"]].copy()

print(f"Reseñas en dataset spoiler: {len(df_sp_labels):,}")
print(f"Con spoiler: {df_sp_labels['has_spoiler'].sum():,}")
print(f"Sin spoiler: {(~df_sp_labels['has_spoiler']).sum():,}")

# Cruzar con nuestra muestra por review_id
df_sample = df_sample.merge(df_sp_labels, on="review_id", how="left")
df_sample["has_spoiler"] = df_sample["has_spoiler"].fillna(False)

print(f"\nReseñas con etiqueta spoiler: {df_sample['has_spoiler'].sum():,}")
print(f"Reseñas sin etiqueta spoiler: {(~df_sample['has_spoiler']).sum():,}")

Cargando dataset de spoilers...


Leyendo spoilers: 0it [00:00, ?it/s]

Reseñas en dataset spoiler: 1,378,033
Con spoiler: 89,672
Sin spoiler: 1,288,361

Reseñas con etiqueta spoiler: 1,949
Reseñas sin etiqueta spoiler: -101,949


In [8]:
# Cargar dataset de spoilers y extraer has_spoiler
print("Cargando dataset de spoilers...")

records_sp = []
with open(SPOILER_FILE, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Leyendo spoilers"):
        records_sp.append(json.loads(line))

df_sp_full = pd.DataFrame(records_sp)

# Extraer has_spoiler del texto
df_sp_full["has_spoiler"] = df_sp_full["review_text"].str.contains(
    r"\(hide spoiler\)", case=False, regex=True, na=False
)

# Quedarnos solo con review_id y has_spoiler
# drop_duplicates por si hay review_ids repetidos
df_sp_labels = df_sp_full[["review_id", "has_spoiler"]].drop_duplicates(
    subset="review_id"
).copy()

print(f"Reseñas únicas en dataset spoiler: {len(df_sp_labels):,}")
print(f"Con spoiler: {df_sp_labels['has_spoiler'].sum():,}")

# Cruzar con nuestra muestra
df_sample = df_sample.drop(columns=["has_spoiler"], errors="ignore")
df_sample = df_sample.merge(df_sp_labels, on="review_id", how="left")
df_sample["has_spoiler"] = df_sample["has_spoiler"].fillna(False)

print(f"\nTotal filas después del merge: {len(df_sample):,}")
print(f"Reseñas CON spoiler: {df_sample['has_spoiler'].sum():,}")
print(f"Reseñas SIN spoiler: {(~df_sample['has_spoiler']).sum():,}")

# Verificar el merge
print(f"Shape del dataframe: {df_sample.shape}")
print(f"Tipos de datos has_spoiler: {df_sample['has_spoiler'].dtype}")
print(f"Valores únicos has_spoiler: {df_sample['has_spoiler'].unique()}")
print(f"Value counts:")
print(df_sample["has_spoiler"].value_counts())

Cargando dataset de spoilers...


Leyendo spoilers: 0it [00:00, ?it/s]

Reseñas únicas en dataset spoiler: 1,378,033
Con spoiler: 89,672

Total filas después del merge: 100,000
Reseñas CON spoiler: 1,949
Reseñas SIN spoiler: -101,949
Shape del dataframe: (100000, 12)
Tipos de datos has_spoiler: object
Valores únicos has_spoiler: [False True]
Value counts:
has_spoiler
False    98051
True      1949
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split

# Split 70% train / 15% val / 15% test
# Estratificado por rating para mantener balance

df_sample["has_spoiler"] = df_sample["has_spoiler"].astype(bool)

train, temp = train_test_split(
    df_sample,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=df_sample["rating"]
)

val, test = train_test_split(
    temp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temp["rating"]
)

print("=== Data Split ===")
print(f"Train: {len(train):,} ({len(train)/len(df_sample)*100:.0f}%)")
print(f"Val:   {len(val):,} ({len(val)/len(df_sample)*100:.0f}%)")
print(f"Test:  {len(test):,} ({len(test)/len(df_sample)*100:.0f}%)")

print("\n=== Distribución ratings en train ===")
print(train["rating"].value_counts().sort_index())

# Guardar los splits
train.to_csv("train.csv", index=False)
val.to_csv("val.csv", index=False)
test.to_csv("test.csv", index=False)

print("\nArchivos guardados: train.csv, val.csv, test.csv ✓")

=== Data Split ===
Train: 70,000 (70%)
Val:   15,000 (15%)
Test:  15,000 (15%)

=== Distribución ratings en train ===
rating
1    14000
2    14000
3    14000
4    14000
5    14000
Name: count, dtype: int64

Archivos guardados: train.csv, val.csv, test.csv ✓
